In [ ]:
import os, sys
from pathlib import Path
import numpy as np

if not '__file__' in dir():
    __file__ = str(Path(os.getcwd())/'notebook.ipynb')  # Convert Path to string
base_dir = os.path.dirname(os.path.abspath(__file__)) # The base_dir variable points to the current directory
parent_dir = os.path.dirname(base_dir)  # Get parent directory
sys.path.append(parent_dir)  # Add the parent directory to the Python path
sys.path.append('../src')

# %load_ext autoreload
%reload_ext autoreload
%autoreload 2

# import warnings
# warnings.filterwarnings('ignore')

# import torch
# torch.set_float32_matmul_precision('high')

import torch
torch.cuda.empty_cache()

In [ ]:
"""load data"""

from data.data_process import prepare_data_wecc

fault_idx = [1, 2, 3, 8, 9, 12, 14, 16, 17, 21, 25, 26, 27, 29, 30, 31, 35, 36, 37, 38, 39, 40, 
            41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 
            61, 62, 63, 66, 67, 70, 72, 73, 74, 83, 98, 99, 100, 101, 102, 103, 104, 105, 
            106, 108, 109, 111, 112, 115, 116, 117, 123, 125, 128, 133, 135, 136, 137, 
            140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 153, 154, 155, 
            156, 157, 159, 160, 162, 163, 164, 165, 166]
gen_buses = [3, 5, 8, 10, 12, 14, 17, 29, 34, 35, 39, 42, 44, 46, 64, 69, 76, 78, 102, 111, 115, 
             117, 137, 139, 143, 147, 148, 158, 161]
data_folder_path = os.path.abspath(os.path.join(base_dir, ".."))+os.sep+'data'
N_traj = len(fault_idx)

# data12 = prepare_data(data_folder_path, fault_idx, 'ieee14_fault_1.2', 1.2)
data = prepare_data_wecc(data_folder_path, fault_idx, 'wecc_full_20', 1.1, phasor_buses = gen_buses)

omega_list =  data['omega_list']
delta_list =  data['delta_list']
time_list =  data['time_list']

# 数据集划分
np.random.seed(0)  # 设置随机种子以确保可重复性
all_indices = np.arange(N_traj)
np.random.shuffle(all_indices)

# 划分数据集
n_train = 90  # 训练集大小

# 获取训练和测试集的索引
train_indices = all_indices[:n_train ].tolist()  # 前用于训练
test_indices = all_indices[n_train:].tolist()    # 后用于测试
train_Ntraj = len(train_indices)
test_Ntraj = len(test_indices)

# 创建输入数据列表
# 重塑数据为 Shape: [n_timesteps, n_features] * n_trajectories
train_x = [np.concatenate((
    (omega_list[ii]-1)*2*np.pi*60,
    delta_list[ii],
    # data['pe_list'][ii],
    # data['voltage_list'][ii]
), axis=1) for ii in train_indices]          
n_vars_x = train_x[0].shape[-1]                   # = omega + delta + pe

# 重塑数据为  [ n_timesteps,n_features] * n_trajectories 
test_x = [np.concatenate((
    (omega_list[ii]-1)*2*np.pi*60,
    delta_list[ii],
    # data['pe_list'][ii],
    # data['voltage_list'][ii]
), axis=1) for ii in test_indices]          # Shape: [n_timesteps, n_features] * n_trajectories

# 打印数据集划分信息
print(f"Total trajectories: {N_traj}")
print(f"Training trajectories per fold: {len(train_indices)}")
# print(f"Test trajectories: {len(test_indices)}")

# 同样处理时间数据
train_time = [time_list[ii] for ii in train_indices]
test_time = [time_list[ii] for ii in test_indices]

# 【添加噪声】
from innkoopman.utils import add_noise_by_snr_numpy
train_x_noise = [None]*(len(train_x))
for ii in range(len(train_x)):
    train_x_noise[ii] = add_noise_by_snr_numpy(train_x[ii], 10)
    train_x_noise[ii] = train_x_noise[ii]


from innkoopman.utils import add_noise_by_snr_numpy
test_x_noise = [None]*(len(test_x))
for ii in range(len(test_x)):
    test_x_noise[ii] = add_noise_by_snr_numpy(test_x[ii], 10)
    test_x_noise[ii] = test_x_noise[ii]

# 打印数据形状以验证
print("训练数据形状:", train_x[0].shape)  # [n_timesteps, n_features] * n_train_trajectories
# print("测试数据形状:", test_x[0].shape)   # [n_timesteps,n_features] * n_test_trajectories

print("训练数据轨迹数量:", len(train_x))
# print("测试数据轨迹数量:", len(test_x))

In [ ]:
from innkoopman.utils import AcademicPlot
# Instantiate the ploting class
plotter = AcademicPlot(style='seaborn-v0_8-paper', figsize=(2, 1.5))
# plotter = AcademicPlot(style=['ieee'], figsize=(4, 3))

In [ ]:
from innkoopman import DeepKoopman

# look_forward = 128
data_dt = data['time_list'][0][1]-data['time_list'][0][0]
state_dim = 58
time_delay_embed = 3
extension_dim = 0


# Create the two-stage model
dlk_regressor = DeepKoopman(
    dt=data_dt,
    look_forward=50,
    time_delay=time_delay_embed,
    config_inn=dict(
        input_size=state_dim*time_delay_embed,
        hidden_size=64*time_delay_embed,
        num_layers=6,
        output_size=state_dim*time_delay_embed + extension_dim,
        init_identity=True,
    ),
    batch_size=256,
    normalize=True,
    trainer_kwargs={'max_epochs': 200, 
                    'accelerator': 'gpu'},

)

# Create directory to save training results
results_folder_path = os.path.abspath(os.path.join(base_dir, ".."))+os.sep+'output'+os.sep + 'WECC_1_snr10'
os.makedirs(results_folder_path, exist_ok=True)

# Train the model
dlk_regressor.fit(
    x=train_x,
    y=None,
    monitor_params=True,
    log_dir=results_folder_path,
    record_losses=True
)

In [ ]:
# torch.save(dlk_regressor._regressor.state_dict(),os.path.join('results_folder_path','model_weights.pth'))

In [ ]:
rmse_error = [None]*train_Ntraj
rrmse_factor = [None]*train_Ntraj
for ii in range(train_Ntraj):
    x0 = train_x[ii][0:time_delay_embed].reshape(-1)  #np.array([2, -4])
    t = train_time[ii]-train_time[ii][0]
    sol = dlk_regressor.simulate(x0[np.newaxis, :], n_steps=len(t)-1)[:,:state_dim] # shape:(sequences, feature)

    rmse_error[ii]  = np.linalg.norm(train_x[ii][:,:state_dim] - sol.real)**2 
    rrmse_factor[ii] = np.linalg.norm(train_x[ii][:,:state_dim])**2

print("误差大小：", np.sqrt(sum(rmse_error)/sum(rrmse_factor))*100, "%")

In [ ]:
# RRMSE: 33.46227351993573  % including the phase               HODMD  %
# RRMSE: 21.28939856027045  % only including the frequency      HODMD  %

Test on training data

In [ ]:
train_idx = 40

x0 = train_x[train_idx][0:time_delay_embed].reshape(-1)  #np.array([2, -4])
T = 10
t = np.arange(0, T, data_dt)

X_train = train_x[train_idx][:len(t)]
# X_train = np.vstack([x0[np.newaxis, :], X_train])

Xkoop_nn = dlk_regressor.simulate(x0[np.newaxis, :], n_steps=len(t)-1)

sol = Xkoop_nn/2/np.pi/60 + 1
train_traj_idx = train_indices[train_idx]
time_train_complete =  data['origin_time'][train_traj_idx]
omega_train_complete = data['origin_omega'][train_traj_idx]

# 绘制曲线
for var_id in [0, 1, 2]:
    scatter_points = [
        {
            'x': train_time[train_idx],
            'y': train_x[train_idx][:,var_id]/2/np.pi/60 + 1,
            'label': 'Noise',
             'color': 'green',
            'marker': '*',
            'size': 2
        }]
    plotter.plot(
        [time_train_complete,            t+train_time[train_idx][0]],
        [omega_train_complete[:,var_id], sol[:,var_id].real   ], 
        scatter_data = scatter_points,
        labels=[r"Truth", r"Deep DMD"], 
        xlabel="Time (s)", 
        ylabel="Omega of Gen \n on Bus " + str(gen_buses[var_id]) + " (p.u.)", 
        title=None, 
        linestyles = ['-','-'],
        filename=None,
        xlim=(0, 10),                # 自定义横坐标范围
        xticks=np.linspace(0, 10, 6), # 自定义横坐标标签
        plot_legend = True,
        plot_grid= False
    )

Testing on unseen trajectory

In [ ]:
test_idx = 0
x0 = test_x[test_idx][0:time_delay_embed].reshape(-1)  #np.array([2, -4])
T = 10
t = np.arange(0, T, data_dt)
Xtest = test_x[test_idx][:len(t)]
# Xtest = np.vstack([x0[np.newaxis, :], Xtest])

Xkoop_nn = dlk_regressor.simulate(x0[np.newaxis, :], n_steps=len(t)-1)

sol = Xkoop_nn/2/np.pi/60 + 1
test_traj_idx = test_indices[test_idx]
time_test_complete =  data['origin_time'][test_traj_idx]
omega_test_complete = data['origin_omega'][test_traj_idx]

# 绘制曲线
for var_id in [0,1,2]:
    plotter.plot(
        [time_test_complete,            t+test_time[test_idx][0]],
        [omega_test_complete[:,var_id], sol[:,var_id].real   ], 
        # scatter_data = scatter_points,
        labels=[r"Truth", r"Deep DMD"], 
        xlabel="Time (s)", 
        ylabel="Omega of Gen \n on Bus " + str(gen_buses[var_id]) + " (p.u.)", 
        title=None, 
        linestyles = ['-','-'],
        filename=None,
        xlim=(0, 10),                # 自定义横坐标范围
        xticks=np.linspace(0, 10, 6), # 自定义横坐标标签
        # plot_legend = True,
        plot_grid= False
    )

In [ ]:
test_idx = 5
x0 = test_x[test_idx][0:time_delay_embed].reshape(-1)  #np.array([2, -4])
T = 10
t = np.arange(0, T, data_dt)
Xtest = test_x[test_idx][:len(t)]
# Xtest = np.vstack([x0[np.newaxis, :], Xtest])

Xkoop_nn = dlk_regressor.simulate(x0[np.newaxis, :], n_steps=len(t)-1)

sol = Xkoop_nn/2/np.pi/60 + 1
test_traj_idx = test_indices[test_idx]
time_test_complete =  data['origin_time'][test_traj_idx]
omega_test_complete = data['origin_omega'][test_traj_idx]

# 绘制曲线
for var_id in [0,1,2]:
    plotter.plot(
        [time_test_complete,            t+test_time[test_idx][0]],
        [omega_test_complete[:,var_id], sol[:,var_id].real   ], 
        # scatter_data = scatter_points,
        labels=[r"Truth", r"Deep DMD"], 
        xlabel="Time (s)", 
        ylabel="Omega of Gen \n on Bus " + str(gen_buses[var_id]) + " (p.u.)", 
        title=None, 
        linestyles = ['-','-'],
        filename=None,
        xlim=(0, 10),                # 自定义横坐标范围
        xticks=np.linspace(0, 10, 6), # 自定义横坐标标签
        # plot_legend = True,
        plot_grid= False
    )

In [ ]:
test_idx = 8
x0 = test_x[test_idx][0:time_delay_embed].reshape(-1)  #np.array([2, -4])
T = 10
t = np.arange(0, T, data_dt)
Xtest = test_x[test_idx][:len(t)]
# Xtest = np.vstack([x0[np.newaxis, :], Xtest])

Xkoop_nn = dlk_regressor.simulate(x0[np.newaxis, :], n_steps=len(t)-1)

sol = Xkoop_nn/2/np.pi/60 + 1
test_traj_idx = test_indices[test_idx]
time_test_complete =  data['origin_time'][test_traj_idx]
omega_test_complete = data['origin_omega'][test_traj_idx]

# 绘制曲线
for var_id in [0,1,2]:
    plotter.plot(
        [time_test_complete,            t+test_time[test_idx][0]],
        [omega_test_complete[:,var_id], sol[:,var_id].real   ], 
        # scatter_data = scatter_points,
        labels=[r"Truth", r"Deep DMD"], 
        xlabel="Time (s)", 
        ylabel="Omega of Gen \n on Bus " + str(gen_buses[var_id]) + " (p.u.)", 
        title=None, 
        linestyles = ['-','-'],
        filename=None,
        xlim=(0, 10),                # 自定义横坐标范围
        xticks=np.linspace(0, 10, 6), # 自定义横坐标标签
        # plot_legend = True,
        plot_grid= False
    )

In [ ]:
train_idx = 52

x0 = train_x[train_idx][0:time_delay_embed].reshape(-1)  #np.array([2, -4])
T = 20
t = np.arange(0, T, data_dt)

X_train = train_x[train_idx][:len(t)]
# X_train = np.vstack([x0[np.newaxis, :], X_train])

Xkoop_nn = dlk_regressor.simulate(x0[np.newaxis, :], n_steps=len(t)-1)

sol = Xkoop_nn/2/np.pi/60 + 1
train_traj_idx = train_indices[train_idx]
time_train_complete =  data['origin_time'][train_traj_idx]
omega_train_complete = data['origin_omega'][train_traj_idx]

# 绘制曲线
for var_id in [7]:
    scatter_points = [
        {
            'x': train_time[train_idx],
            'y': train_x_noise[train_idx][:,var_id]/2/np.pi/60 + 1,
            'label': 'Noise',
             'color': 'green',
            'marker': '*',
            'size': 2
        }]
    plotter.plot(
        [time_train_complete,            t+train_time[train_idx][0]],
        [omega_train_complete[:,var_id], sol[:,var_id].real   ], 
        scatter_data = scatter_points,
        labels=[r"Truth", r"Deep DMD"], 
        xlabel="Time (s)", 
        ylabel="Omega of Gen \n on Bus " + str(gen_buses[var_id]) + " (p.u.)", 
        title=None, 
        linestyles = ['-','-'],
        filename=None,
        xlim=(0, 20),                # 自定义横坐标范围
        xticks=np.linspace(0, 20, 6), # 自定义横坐标标签
        plot_legend = True,
        plot_grid= False
    )

In [ ]:
# 在测试集上作图
test_idx = 7

x0 = test_x[test_idx][0:time_delay_embed].reshape(-1)  #np.array([2, -4])
T = 10
t = np.arange(0, T, data_dt)
Xtest = test_x[test_idx][:len(t)]

Xkoop_nn = dlk_regressor.simulate(x0[np.newaxis, :], n_steps=len(t)-1)

sol = Xkoop_nn/2/np.pi/60 + 1
test_traj_idx = test_indices[test_idx]
time_test_complete =  data['origin_time'][test_traj_idx]
omega_test_complete = data['origin_omega'][test_traj_idx]

# 绘制曲线
for var_id in [19]:
    plotter.plot(
        [time_test_complete,            t+test_time[test_idx][0]],
        [omega_test_complete[:,var_id], sol[:,var_id].real   ], 
        # scatter_data = scatter_points,
        labels=[r"Truth", r"Deep DMD"], 
        xlabel="Time (s)", 
        ylabel="Omega of Gen \n on Bus " + str(gen_buses[var_id]) + " (p.u.)", 
        title=None, 
        linestyles = ['-','-'],
        filename=None,
        xlim=(0, 10),                # 自定义横坐标范围
        xticks=np.linspace(0, 10, 6), # 自定义横坐标标签
        # plot_legend = True,
        plot_grid= False
    )

In [ ]:
psi_0 = dlk_regressor._compute_psi(x0)  # (n_koopman, n_samples)
T = 10      
t =  np.arange(0, T, data_dt)
# Advance using eigenvalues
psi_t = np.zeros((state_dim*time_delay_embed+extension_dim, len(t)))                  # (n_koopman, n_samples)
for tt in range(len(t)):
    psi_t[:, tt] = (np.diag(np.power(dlk_regressor._eigenvalues_, tt)) @ psi_0)[:,-1]  # (n_koopman, n_samples)

In [ ]:
# 绘制曲线
for var_id in [6]:
    plotter.plot(
        [t],
        [psi_t[1]], 
        labels=[r"Deep DMD"], 
        xlabel="Time (s)", 
        title=None, 
        linestyles = ['-'],
        filename=None,
        xlim=(0, T),                # 自定义横坐标范围
        xticks=np.linspace(0, 10, 6), # 自定义横坐标标签
        plot_legend = True,
        plot_grid= False
    )

In [ ]:
abs(dlk_regressor.eigenvalues_)

In [ ]:
from innkoopman.utils import plot_eigs
plot_eigs(dlk_regressor.eigenvalues_)

In [ ]:
eigv = np.log(dlk_regressor.eigenvalues_)/data_dt
print(eigv)

In [ ]:
mode_i =0
tmp = dlk_regressor._eigenvectors_[:,[mode_i]] @ dlk_regressor.psi(train_x[train_idx][0:time_delay_embed].reshape(-1))[[mode_i],:]
print(tmp)
print(np.linalg.norm(tmp))

In [ ]:
dlk_regressor._eigenvectors_[:,[0]] @ dlk_regressor._eigenvectors_.T.conj()[[0],:]